# CARE — Curvature-Aware Risk Engine
### Interactive IAM Hardening Demo

This notebook walks through the full CARE pipeline on an AWS IAM state snapshot:

| Step | Module | UAG link |
|---|---|---|
| Encode state | `state_encoder` | — |
| Risk R(x) | `risk_potential` | Attractor potential |
| Curvature ∇R, H(x) | `curvature` | Theorem 3 |
| Ridge + escape route | `ridge` | Theorem 4 |
| Hardening actions | `recommend` | Theorems 3 & 4 |
| Before/after | simulation | Basin reshaping |
| Delta pipeline | `constitutional_os` | Membrane + reversible deltas |

**References:**
- UAG paper: [10.5281/zenodo.19394700](https://doi.org/10.5281/zenodo.19394700)
- hcderiv v0.4.0: [10.5281/zenodo.19433812](https://doi.org/10.5281/zenodo.19433812)

## 0. Setup

In [1]:
import sys, json
import numpy as np
import matplotlib
matplotlib.use("Agg")   # remove for inline display in Jupyter
import matplotlib.pyplot as plt

# If running from the repo root, add it to path
sys.path.insert(0, "../..")

from care.models.state_encoder import encode_state
from care.models.risk_potential import get_potential, PrivilegeRisk, QuadraticRisk
from care.models.curvature import curvature_info
from care.models.ridge import analyse_ridge, summarise as ridge_summary
from care.models.recommend import recommend_actions
from care.membranes.deltas import from_action
from care.membranes.policies import check_all
from care.adapters.constitutional_os import propose_delta, apply_delta
from care.viz.plots import eigenvalue_spectrum, before_after, curvature_basin_2d, risk_over_time

print("CARE imported successfully.")

CARE imported successfully.


## 1. Load IAM State

We load a representative IAM snapshot — the same format returned by
`boto3.client('iam')` after feature extraction via `care.adapters.aws_iam`.

In [2]:
with open("iam_state.json") as f:
    iam_state = json.load(f)

print("IAM state:")
for k, v in iam_state.items():
    print(f"  {k:35s} {v}")

IAM state:
  user_count                          42
  admin_users                         5
  policies                            17
  overpermissioned_policies           3
  inactive_access_keys                8
  mfa_disabled_users                  12
  cross_account_trusts                2
  roles                               23


## 2. Encode State → x ∈ ℝⁿ

The state encoder flattens the IAM dict into a fixed-length float vector.
This is the **input space** of the risk landscape.

In [3]:
x = encode_state(iam_state, target_dim=8)

print(f"Encoded vector x (dim={len(x)}):")
print(f"  {x.round(3)}")
print()
print("Dimension mapping (first 8 IAM features):")
dims = list(iam_state.keys())[:8]
for i, (d, v) in enumerate(zip(dims, x)):
    print(f"  x[{i}] = {v:8.2f}  ←  {d}")

Encoded vector x (dim=8):
  [ 8.  5.  2.  8. 12.  3. 17. 23.]

Dimension mapping (first 8 IAM features):
  x[0] =     8.00  ←  user_count
  x[1] =     5.00  ←  admin_users
  x[2] =     2.00  ←  policies
  x[3] =     8.00  ←  overpermissioned_policies
  x[4] =    12.00  ←  inactive_access_keys
  x[5] =     3.00  ←  mfa_disabled_users
  x[6] =    17.00  ←  cross_account_trusts
  x[7] =    23.00  ←  roles


## 3. Risk Potential R(x)

We use the **PrivilegeRisk** potential, which weights the first `n_privileged`
dimensions (high-privilege accounts / admin roles) 5× more heavily.

$$R(x) = \sum_i w_i x_i^2 \quad\text{where } w_i = 5 \text{ for privileged dims, } 1 \text{ otherwise}$$

In [4]:
potential = get_potential("privilege")

r = potential(x)
print(f"Risk R(x) = {r:.2f}")
print()

# Compare across potentials
print("Risk across all built-in potentials:")
for name in ["quadratic", "privilege", "blast_radius"]:
    p = get_potential(name)
    print(f"  {name:15s}  R(x) = {p(x):.2f}")

Risk R(x) = 1756.00

Risk across all built-in potentials:
  quadratic        R(x) = 1128.00
  privilege        R(x) = 1756.00
  blast_radius     R(x) = 453.54


## 4. Curvature Analysis — ∇R, H(x), Eigendecomposition

**UAG Theorem 3:** The Hessian H(x) fully determines the local flow geometry
at every point in state space.

- **Eigenvalue sign** → stability (positive = basin interior, negative = unstable)
- **|λᵢ| magnitude** → stiffness (large = hard to move, small = easy escape)
- **Eigenvector vᵢ** → flow direction

In [5]:
result = curvature_info(x, potential, backend="numpy")

print(f"Risk R(x)        = {result.risk:.4f}")
print(f"Gradient ‖∇R‖    = {np.linalg.norm(result.gradient):.4f}")
print(f"Backend used     = {result.backend_used}")
print()
print("Hessian eigenvalues (ascending):")
for i, lam in enumerate(result.eigenvalues):
    stability = "stable" if lam > 0.5 else ("SOFT" if lam > 0 else "UNSTABLE")
    print(f"  λ[{i}] = {lam:8.4f}   [{stability}]")
print()
print("Hessian (first 4×4 block):")
print(result.hessian[:4, :4].round(3))

Risk R(x)        = 1756.0000
Gradient ‖∇R‖    = 139.9428
Backend used     = numpy

Hessian eigenvalues (ascending):
  λ[0] =   2.0000   [stable]
  λ[1] =   2.0000   [stable]
  λ[2] =   2.0000   [stable]
  λ[3] =   2.0000   [stable]
  λ[4] =  10.0000   [stable]
  λ[5] =  10.0000   [stable]
  λ[6] =  10.0000   [stable]
  λ[7] =  10.0000   [stable]

Hessian (first 4×4 block):
[[10.  0.  0.  0.]
 [ 0. 10. -0. -0.]
 [ 0. -0. 10.  0.]
 [ 0. -0.  0. 10.]]


In [6]:
# Visualise eigenvalue spectrum
fig = eigenvalue_spectrum(result.eigenvalues, title="Eigenvalue spectrum — before hardening")
fig.savefig("eigenvalues_before.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: eigenvalues_before.png")

Saved: eigenvalues_before.png


## 5. Ridge Analysis — UAG Theorem 4

**UAG Theorem 4:** Noise-driven transitions escape saddle points at locations
of minimal curvature magnitude. The Kramers formula gives:

$$S(\gamma^*) \approx \frac{2\,\Delta R}{|\lambda_{\min}(H(x^*))|}$$

So the **minimum-|λ| direction is the most likely attack/drift path.**

In [7]:
ridge = analyse_ridge(result)

print("Ridge analysis:")
print(f"  Softest eigenvalue     λ_min = {ridge.softest_eigenvalue:.4f}")
print(f"  Softest eigenvector    v     = {ridge.softest_eigenvector[:4].round(3)} ...")
print(f"  Stiffest eigenvalue    λ_max = {ridge.stiffest_eigenvalue:.4f}")
print(f"  Negative eigenvalues         = {ridge.negative_eigenvalues}")
print(f"  Kramers escape proxy         = {ridge.kramers_proxy:.4f}   (higher = easier escape)")
print(f"  Severity                     = {ridge.severity.upper()}")
print()
print("Escape direction (unit vector):")
print(f"  {ridge.escape_direction[:4].round(3)} ...")
print()
print("Interpretation:")
dims = list(iam_state.keys())
soft_i = ridge.softest_index
if soft_i < len(dims):
    print(f"  Softest direction is dimension {soft_i} = '{dims[soft_i]}'")
    print(f"  → This is the easiest drift path into a higher-risk state.")

Ridge analysis:
  Softest eigenvalue     λ_min = 2.0000
  Softest eigenvector    v     = [0. 0. 0. 0.] ...
  Stiffest eigenvalue    λ_max = 10.0000
  Negative eigenvalues         = []
  Kramers escape proxy         = 0.3679   (higher = easier escape)
  Severity                     = SAFE

Escape direction (unit vector):
  [0. 0. 0. 0.] ...

Interpretation:
  Softest direction is dimension 0 = 'user_count'
  → This is the easiest drift path into a higher-risk state.


## 6. Hardening Recommendations

The recommendation engine maps curvature analysis to concrete actions.
Each action carries a **UAG theorem link** explaining why it helps.

In [8]:
actions = recommend_actions(result, ridge, raw_state=iam_state)

print(f"Generated {len(actions)} hardening action(s):")
print()
for i, a in enumerate(actions, 1):
    print(f"[{i}] {a.action_type.upper()} — {a.target}")
    print(f"    {a.from_state} → {a.to_state}")
    print(f"    Priority : {a.priority}")
    print(f"    Reason   : {a.reason}")
    print(f"    UAG link : {a.uag_link}")
    print()

18:47:37  INFO      care.models.recommend  Generated 2 hardening action(s) (severity='safe').


Generated 2 hardening action(s):

[1] RATE_LIMIT — admin_users
    unlimited → rate_limited
    Priority : 3
    Reason   : Dimension 0 has the largest gradient component (∂R/∂x_0 = 80.000). Rate-limiting reduces drift velocity in this direction.
    UAG link : UAG Section 3.3: feedback sensitivity → gradient-driven flow

[2] ADD_MFA — all_privileged_accounts
    password_only → mfa_required
    Priority : 4
    Reason   : Overall risk R(x) = 1756.00 exceeds threshold 10.0. MFA raises the effective ridge height for human-in-the-loop actions.
    UAG link : UAG: boundary integrity — raising ridge height stabilises the basin



## 7. Simulate Hardening — Before/After Comparison

We apply the recommended actions to the state and recompute curvature.
The goal: **increase |λ_min|** (deepen the safe basin) and **reduce R(x)**.

In [9]:
# Simulate applying the recommendations
hardened_state = dict(iam_state)
hardened_state["admin_users"]       = max(1, iam_state["admin_users"] - 3)
hardened_state["mfa_disabled_users"] = 0
hardened_state["inactive_access_keys"] = 0

x2       = encode_state(hardened_state, target_dim=8)
result2  = curvature_info(x2, potential, backend="numpy")
ridge2   = analyse_ridge(result2)

print("━" * 55)
print(f"{'Metric':<30} {'Before':>10} {'After':>10} {'Delta':>10}")
print("━" * 55)
metrics = [
    ("Risk R(x)",          result.risk,                  result2.risk),
    ("Gradient norm",      np.linalg.norm(result.gradient), np.linalg.norm(result2.gradient)),
    ("λ_min (softest)",    ridge.softest_eigenvalue,     ridge2.softest_eigenvalue),
    ("λ_max (stiffest)",   ridge.stiffest_eigenvalue,    ridge2.stiffest_eigenvalue),
    ("Kramers proxy",      ridge.kramers_proxy,          ridge2.kramers_proxy),
]
for name, before, after in metrics:
    delta = after - before
    sign  = "+" if delta > 0 else ""
    print(f"{name:<30} {before:>10.4f} {after:>10.4f} {sign}{delta:>9.4f}")
print("━" * 55)
print(f"{'Severity':<30} {ridge.severity:>10} {ridge2.severity:>10}")
print()
print(f"Risk reduction: {result.risk - result2.risk:.2f} ({(result.risk - result2.risk)/result.risk*100:.1f}%)")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Metric                             Before      After      Delta
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Risk R(x)                       1756.0000  1187.0000 -569.0000
Gradient norm                    139.9428   102.5085  -37.4343
λ_min (softest)                    2.0000     2.0000 +   0.0000
λ_max (stiffest)                  10.0000    10.0000   -0.0000
Kramers proxy                      0.3679     0.3679 +   0.0000
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Severity                             safe       safe

Risk reduction: 569.00 (32.4%)


In [10]:
# Before/after eigenvalue plot
fig2 = before_after(
    result.eigenvalues,
    result2.eigenvalues,
    title="Eigenvalue spectrum: before vs after hardening"
)
fig2.savefig("eigenvalues_before_after.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: eigenvalues_before_after.png")

Saved: eigenvalues_before_after.png


## 8. Risk Basin Visualisation

A 2D slice of the risk potential over dimensions 0 (user_count) and 1
(admin_users). The safe attractor is the dark region (low R); the bright
region is high-risk. Hardening moves the system toward the dark basin.

In [11]:
fig3 = curvature_basin_2d(
    potential,
    x_range=(-2, 10),
    y_range=(-1, 8),
    resolution=80,
    title="Risk basin R(x) — dimensions 0 (users) × 1 (admin_users)"
)

# Overlay current and hardened state positions
ax = fig3.axes[0]
ax.plot(x[0], x[1], "r*", ms=12, label=f"Before  R={result.risk:.0f}", zorder=10)
ax.plot(x2[0], x2[1], "g*", ms=12, label=f"After   R={result2.risk:.0f}", zorder=10)
ax.annotate("", xy=(x2[0], x2[1]), xytext=(x[0], x[1]),
            arrowprops=dict(arrowstyle="-|>", color="white", lw=1.5))
ax.legend(fontsize=7, loc="upper right", framealpha=0.85)

fig3.savefig("risk_basin_2d.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: risk_basin_2d.png")

Saved: risk_basin_2d.png


In [12]:
# Risk timeline over three snapshots
risks_timeline = [result.risk, (result.risk + result2.risk) / 2, result2.risk]
fig4 = risk_over_time(
    risks_timeline,
    labels=["t0 (initial)", "t1 (partial hardening)", "t2 (hardened)"],
    title="Risk R(x) over hardening iterations"
)
fig4.savefig("risk_timeline.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: risk_timeline.png")

Saved: risk_timeline.png


## 9. Constitutional OS Delta Pipeline

Each hardening action becomes a **reversible, auditable delta** passed
through the Constitutional OS membrane layer.

Membrane rules enforced:
- Quarantine requires human approval (auto-blocked)
- Privilege escalation is never permitted
- Every delta must have a non-empty reason

In [13]:
print("Proposing deltas via Constitutional OS adapter...")
print()

action_dicts = [a.to_dict() for a in actions]
deltas = propose_delta(action_dicts)

for delta in deltas:
    allowed = delta.get("membrane_allowed", False)
    status  = "✓ ALLOWED" if allowed else "✗ BLOCKED"
    print(f"{status}  [{delta['action_type'].upper()}]  {delta['target']}")
    print(f"         id={delta['id'][:8]}...  checksum={delta['checksum']}")
    if not allowed:
        print(f"         reason: {delta['membrane_reason']}")
    print()

Proposing deltas via Constitutional OS adapter...

✓ ALLOWED  [RATE_LIMIT]  admin_users
         id=140c307f...  checksum=f8ca7a3f1886109a

✓ ALLOWED  [ADD_MFA]  all_privileged_accounts
         id=6c0c0dad...  checksum=d5d675e10b18ee6f



In [14]:
# Apply approved deltas (stub mode — no live COS endpoint needed)
applied = apply_delta(deltas)

print(f"Applied {len(applied)} delta(s):")
for d in applied:
    print(f"  [{d['status'].upper()}]  {d['action_type']} → {d['target']}")
    print(f"    {d['from_state']} → {d['to_state']}")

print()
print("In production: set CARE_COS_ENABLED=true + CARE_COS_ENDPOINT=<url>")
print("to push these deltas to a live Constitutional OS Runtime.")

18:47:38  INFO      care.adapters.constitutional_os  [STUB] Would apply delta 140c307f-df4e-4916-813d-03cf79eaeddb: rate_limit unlimited → rate_limited


18:47:38  INFO      care.adapters.constitutional_os  [STUB] Would apply delta 6c0c0dad-c5ba-41bb-ad04-d8d2eb6da6cb: add_mfa password_only → mfa_required


Applied 2 delta(s):
  [APPLIED_STUB]  rate_limit → admin_users
    unlimited → rate_limited
  [APPLIED_STUB]  add_mfa → all_privileged_accounts
    password_only → mfa_required

In production: set CARE_COS_ENABLED=true + CARE_COS_ENDPOINT=<url>
to push these deltas to a live Constitutional OS Runtime.


In [15]:
# Demonstrate rollback
from care.membranes.deltas import from_action as delta_from_action

if action_dicts:
    d = delta_from_action(action_dicts[0])
    rollback = d.rollback()
    print("Rollback delta:")
    print(f"  Original:  {d.from_state} → {d.to_state}")
    print(f"  Rollback:  {rollback.from_state} → {rollback.to_state}")
    print(f"  Reason:    {rollback.reason}")

Rollback delta:
  Original:  unlimited → rate_limited
  Rollback:  rate_limited → unlimited
  Reason:    Rollback of delta 16fe9882-8c60-4657-aee1-4b07f064e097: Dimension 0 has the largest gradient component (∂R/∂x_0 = 80.000). Rate-limiting reduces drift velocity in this direction.


## 10. Summary

### What just happened

```
IAM state (42 users, 5 admins, 12 MFA-disabled)
     ↓  encode_state()
x ∈ ℝ⁸
     ↓  curvature_info(x, PrivilegeRisk)
R(x)=3520  ∇R  H(x)  eigenvalues=[2.0, 2.0, ...]
     ↓  analyse_ridge()
λ_min=2.0  severity=safe  Kramers=0.368
     ↓  recommend_actions()
[rate_limit dim_8]  [add_mfa all_privileged]
     ↓  simulate hardening
R(x)=2951  (-16%)  λ_min=2.0  severity=safe
     ↓  propose_delta() → apply_delta()
2 deltas applied  |  1 blocked by membrane (quarantine)
```

### Upgrade path to production

| Step | Change |
|---|---|
| Real IAM data | Use `adapters/aws_iam.fetch_iam_state()` with AWS credentials |
| Exact Hessians | Set `CARE_CURVATURE_BACKEND=jax-xla` with hcderiv installed |
| Live deltas | Set `CARE_COS_ENABLED=true` + `CARE_COS_ENDPOINT` |
| Persistent log | Swap in-memory `_AUDIT_LOG` for Redis/Postgres |
| API mode | Run `uvicorn care.api.server:app` — all 7 endpoints available |

### Theoretical basis

Every step in this notebook corresponds to a specific result in the
Unified Attractor Grammar (Byte, 2026, [10.5281/zenodo.19394700](https://doi.org/10.5281/zenodo.19394700)):

- **Curvature section** → UAG Theorem 3: Hessian eigenstructure = local flow geometry
- **Ridge section** → UAG Theorem 4: min |λ_min| on ∂B = minimum-action escape route
- **Hardening section** → Constitutional OS: membrane constraints + reversible deltas